# Bengali MMS CTC final confirmed tiny-overfit test

This is the final post-audit test notebook. It uses the attached local
four-dialect dataset and saved checkpoint, records the user's confirmation for
all 32 audio/transcript pairs, validates the checkpoint contract, and runs a
fresh plain MMS-CTC overfit test. No MoE, dialect loss, augmentation, or DDP is
used. Existing datasets, checkpoints, and earlier notebook outputs are not
modified.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

REPO_COMMIT = "4cb0d20"
REPO_DIR = Path("/kaggle/working/bengali-dialect-asr")
RUN_DIR = Path("/kaggle/working/ctc-collapse-diagnostics")
LOG_DIR = RUN_DIR / "logs"
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Recreate only the small source snapshot needed by the diagnostics. This is
# deliberately local: Kaggle workers sometimes cannot resolve github.com even
# when the notebook has internet enabled.
EMBEDDED_SOURCES = {'scripts/ctc_collapse_diagnostics.py': '#!/usr/bin/env python\n"""Audit CTC wiring, labels, lengths, logits, and decoding for one checkpoint.\n\nThis script is read-only with respect to the dataset and checkpoint.  It writes\nJSON/CSV reports below ``--output-dir`` and never starts training.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport numpy as np\nimport os\nimport random\nimport re\nimport sys\nimport unicodedata\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path, PurePosixPath\nfrom typing import Iterable, Iterator\n\nEXPECTED_BLANK_ID = 0\nEXPECTED_DELIMITER_ID = 2\nEXPECTED_VOCAB_SIZE = 73\nTARGET_SAMPLE_RATE = 16_000\n\nDISTRICT_TO_DIALECT = {\n    "Alipurduar": "Kamrupi",\n    "CoochBehar": "Kamrupi",\n    "Darjeeling": "Kamrupi",\n    "Jalpaiguri": "Kamrupi",\n    "Jhargram": "Jharkhandi",\n    "PaschimMedinipur": "Jharkhandi",\n    "Purulia": "Jharkhandi",\n    "Malda": "Varendri",\n    "DakshinDinajpur": "Varendri",\n    "North24Parganas": "Rarhi",\n    "Kolkata": "Rarhi",\n}\n\nZERO_WIDTH_RE = re.compile(r"[\\u200B-\\u200D\\uFEFF]")\nWHITESPACE_RE = re.compile(r"\\s+")\n\n\ndef normalize_text(value: object) -> str:\n    text = unicodedata.normalize("NFC", str(value or ""))\n    text = ZERO_WIDTH_RE.sub("", text)\n    cleaned = []\n    for character in text:\n        if character.isspace():\n            cleaned.append(" ")\n        elif "\\u0980" <= character <= "\\u09FF" and unicodedata.category(character)[0] in {"L", "M", "N"}:\n            cleaned.append(character)\n        else:\n            cleaned.append(" ")\n    return WHITESPACE_RE.sub(" ", "".join(cleaned)).strip()\n\n\ndef split_layout(root: Path) -> str:\n    if all((root / split).is_dir() for split in ("train", "validation", "test")):\n        return "directories"\n    if all((root / f"{split}.zip").is_file() for split in ("train", "validation", "test")):\n        return "split-zips"\n    raise RuntimeError(\n        f"{root} must contain train/validation/test directories or split ZIP files"\n    )\n\n\ndef iter_pairs(root: Path, split: str) -> Iterator[dict]:\n    """Yield matched WAV/TXT records for exactly the 11 approved districts."""\n    mode = split_layout(root)\n    if mode == "directories":\n        split_dir = root / split\n        for district in DISTRICT_TO_DIALECT:\n            district_dir = split_dir / district\n            if not district_dir.is_dir():\n                raise RuntimeError(f"Missing required district directory: {district_dir}")\n            txt_by_key = {\n                path.relative_to(district_dir).with_suffix("").as_posix(): path\n                for path in district_dir.rglob("*.txt")\n            }\n            wav_by_key = {\n                path.relative_to(district_dir).with_suffix("").as_posix(): path\n                for path in district_dir.rglob("*.wav")\n            }\n            if set(txt_by_key) != set(wav_by_key):\n                raise RuntimeError(\n                    f"WAV/TXT mismatch in {district}: "\n                    f"missing_audio={len(set(txt_by_key) - set(wav_by_key))} "\n                    f"missing_text={len(set(wav_by_key) - set(txt_by_key))}"\n                )\n            for key in sorted(txt_by_key):\n                transcript = normalize_text(\n                    txt_by_key[key].read_text(encoding="utf-8-sig", errors="strict")\n                )\n                if transcript:\n                    yield {\n                        "sample_id": f"{district}/{key}",\n                        "audio": wav_by_key[key],\n                        "transcript": transcript,\n                        "district": district,\n                        "dialect": DISTRICT_TO_DIALECT[district],\n                    }\n        return\n\n    with zipfile.ZipFile(root / f"{split}.zip") as archive:\n        grouped: dict[tuple[str, str], dict[str, str]] = {}\n        for info in archive.infolist():\n            if info.is_dir():\n                continue\n            parts = list(PurePosixPath(info.filename).parts)\n            if parts and parts[0].lower() == split.lower():\n                parts = parts[1:]\n            if len(parts) < 2 or parts[0] not in DISTRICT_TO_DIALECT:\n                continue\n            relative = PurePosixPath(*parts[1:])\n            suffix = relative.suffix.lower()\n            if suffix not in {".wav", ".txt"}:\n                continue\n            grouped.setdefault((parts[0], relative.with_suffix("").as_posix()), {})[\n                suffix\n            ] = info.filename\n        for (district, key), pair in sorted(grouped.items()):\n            if set(pair) != {".wav", ".txt"}:\n                raise RuntimeError(f"ZIP WAV/TXT mismatch: {district}/{key}")\n            transcript = normalize_text(\n                archive.read(pair[".txt"]).decode("utf-8-sig", errors="strict")\n            )\n            if transcript:\n                yield {\n                    "sample_id": f"{district}/{key}",\n                    "audio": (root / f"{split}.zip", pair[".wav"]),\n                    "transcript": transcript,\n                    "district": district,\n                    "dialect": DISTRICT_TO_DIALECT[district],\n                }\n\n\ndef choose_rows(root: Path, split: str, limit: int, seed: int = 42) -> list[dict]:\n    rows = list(iter_pairs(root, split))\n    rows.sort(key=lambda row: row["sample_id"])\n    if limit <= 0 or len(rows) <= limit:\n        return rows\n    groups = {district: [] for district in DISTRICT_TO_DIALECT}\n    for row in rows:\n        groups[row["district"]].append(row)\n    rng = random.Random(seed)\n    for values in groups.values():\n        rng.shuffle(values)\n    selected = []\n    while len(selected) < limit:\n        progressed = False\n        for district in DISTRICT_TO_DIALECT:\n            if groups[district] and len(selected) < limit:\n                selected.append(groups[district].pop())\n                progressed = True\n        if not progressed:\n            break\n    selected.sort(key=lambda row: row["sample_id"])\n    return selected\n\n\ndef make_manifest(data_root: Path, output: Path, count: int = 32, seed: int = 42) -> dict:\n    rows = choose_rows(data_root, "test", max(count, len(DISTRICT_TO_DIALECT)), seed)\n    rows = rows[:count]\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(\n            handle,\n            fieldnames=["sample_id", "audio", "transcript", "district", "dialect", "manually_verified"],\n        )\n        writer.writeheader()\n        for row in rows:\n            audio = row["audio"]\n            if isinstance(audio, tuple):\n                audio = f"{audio[0]}::{audio[1]}"\n            writer.writerow({**row, "audio": str(audio), "manually_verified": "NO"})\n    summary = {\n        "manifest": str(output),\n        "count": len(rows),\n        "district_counts": dict(Counter(row["district"] for row in rows)),\n        "requires_manual_audio_transcript_check": True,\n    }\n    output.with_name("tiny_manifest_summary.json").write_text(\n        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    return summary\n\n\ndef read_audio(source, archive_cache: dict | None = None):\n    import io\n    import soundfile as sf\n    import torch\n    import torchaudio\n\n    if isinstance(source, tuple):\n        archive_cache = archive_cache if archive_cache is not None else {}\n        archive = archive_cache.get(str(source[0]))\n        if archive is None:\n            archive = zipfile.ZipFile(source[0])\n            archive_cache[str(source[0])] = archive\n        handle = io.BytesIO(archive.read(source[1]))\n    else:\n        handle = source\n    waveform, sample_rate = sf.read(handle, dtype="float32", always_2d=True)\n    waveform = waveform.mean(axis=1)\n    if not len(waveform) or not np.isfinite(waveform).all():\n        raise ValueError("Audio is empty or contains non-finite samples")\n    if sample_rate != TARGET_SAMPLE_RATE:\n        waveform = torchaudio.functional.resample(\n            torch.from_numpy(waveform), int(sample_rate), TARGET_SAMPLE_RATE\n        ).numpy()\n    return waveform.astype("float32", copy=False)\n\n\ndef _load_processor(checkpoint: Path, model_name: str):\n    from transformers import Wav2Vec2Processor\n\n    candidates = [checkpoint / "processor", checkpoint]\n    parent = checkpoint.parent\n    for _ in range(3):\n        candidates.append(parent / "processor")\n        parent = parent.parent\n    candidate = next(\n        (path for path in candidates if (path / "processor_config.json").is_file()),\n        checkpoint,\n    )\n    processor = Wav2Vec2Processor.from_pretrained(candidate)\n    tokenizer = processor.tokenizer\n    if len(tokenizer) != EXPECTED_VOCAB_SIZE:\n        raise ValueError(f"Expected Bengali CTC vocabulary {EXPECTED_VOCAB_SIZE}, got {len(tokenizer)}")\n    if tokenizer.pad_token_id != EXPECTED_BLANK_ID:\n        raise ValueError(f"Expected blank/pad ID 0, got {tokenizer.pad_token_id}")\n    if tokenizer.unk_token_id != 1:\n        raise ValueError(f"Expected unknown ID 1, got {tokenizer.unk_token_id}")\n    if tokenizer.convert_tokens_to_ids("|") != EXPECTED_DELIMITER_ID:\n        raise ValueError("Expected word delimiter ID 2")\n    if tokenizer.word_delimiter_token != "|":\n        raise ValueError("Expected word delimiter token \'|\'")\n    feature = processor.feature_extractor\n    if int(feature.sampling_rate) != TARGET_SAMPLE_RATE or not bool(feature.do_normalize):\n        raise ValueError("Checkpoint processor is not the normalized 16-kHz MMS processor")\n    return processor\n\n\ndef _load_model(checkpoint: Path, repo_root: Path, processor, model_name: str, device):\n    import torch\n    from omegaconf import OmegaConf\n    from asr_dialect_benchmark.modeling import BengaliDialectASR\n\n    config_path = checkpoint / "config.json"\n    if not config_path.is_file():\n        raise FileNotFoundError(f"Missing checkpoint config: {config_path}")\n    config_data = json.loads(config_path.read_text(encoding="utf-8"))\n    saved_tokens = int(config_data.get("model", {}).get("num_tokens", EXPECTED_VOCAB_SIZE))\n    if saved_tokens != EXPECTED_VOCAB_SIZE:\n        raise ValueError(\n            "Checkpoint model.num_tokens is not the Bengali CTC size: "\n            f"{saved_tokens}"\n        )\n    config = OmegaConf.create(config_data)\n    config.model.num_tokens = EXPECTED_VOCAB_SIZE\n    config.model.gradient_checkpointing = False\n    model = BengaliDialectASR(config)\n    if model.ctc_head.out_features != EXPECTED_VOCAB_SIZE:\n        raise ValueError(f"CTC head has {model.ctc_head.out_features} outputs")\n\n    state_path = None\n    for candidate in ("model.safetensors", "model_state.pt", "pytorch_model.bin"):\n        if (checkpoint / candidate).is_file():\n            state_path = checkpoint / candidate\n            break\n    if state_path is None:\n        raise FileNotFoundError(f"No model state found in {checkpoint}")\n    if state_path.suffix == ".safetensors":\n        from safetensors.torch import load_file\n\n        state = load_file(str(state_path), device="cpu")\n    else:\n        try:\n            state = torch.load(state_path, map_location="cpu", weights_only=True)\n        except TypeError:\n            state = torch.load(state_path, map_location="cpu")\n    if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):\n        state = state["state_dict"]\n    state = {\n        (key.removeprefix("module.") if key.startswith("module.") else key): value\n        for key, value in state.items()\n    }\n    missing, unexpected = model.load_state_dict(state, strict=False)\n    if any(key.startswith("ctc_head.") for key in missing):\n        raise ValueError(f"Checkpoint is missing Bengali CTC head weights: {missing}")\n    model.to(device).eval()\n    return model, config_data, {\n        "state_path": str(state_path),\n        "missing_keys": list(missing),\n        "unexpected_keys": list(unexpected),\n    }\n\n\ndef _decode(processor, ids: Iterable[int], blank_id: int) -> str:\n    collapsed = []\n    previous = None\n    for raw in ids:\n        token_id = int(raw)\n        if token_id == blank_id:\n            previous = token_id\n            continue\n        if token_id == previous:\n            continue\n        collapsed.append(token_id)\n        previous = token_id\n    return processor.tokenizer.decode(\n        collapsed, group_tokens=False, skip_special_tokens=True\n    ).strip()\n\n\ndef edit_distance(reference: list, hypothesis: list) -> int:\n    previous = list(range(len(hypothesis) + 1))\n    for i, ref in enumerate(reference, start=1):\n        current = [i]\n        for j, hyp in enumerate(hypothesis, start=1):\n            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ref != hyp)))\n        previous = current\n    return previous[-1]\n\n\ndef _bias_report(model) -> dict:\n    bias = model.ctc_head.bias.detach().float().cpu()\n    values = bias.tolist()\n    largest = sorted(enumerate(values), key=lambda item: item[1], reverse=True)[:10]\n    return {\n        "blank_bias_id_0": float(values[EXPECTED_BLANK_ID]),\n        "delimiter_bias_id_2": float(values[EXPECTED_DELIMITER_ID]),\n        "largest_biases": [[int(index), float(value)] for index, value in largest],\n        "initialization": "torch.nn.Linear default initialization unless checkpoint metadata says otherwise",\n    }\n\n\ndef audit_checkpoint(\n    checkpoint: Path,\n    data_root: Path,\n    repo_root: Path,\n    output_dir: Path,\n    sample_count: int,\n    batch_size: int,\n    model_name: str,\n    seed: int,\n) -> dict:\n    import torch\n\n    sys.path.insert(0, str(repo_root / "src"))\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    processor = _load_processor(checkpoint, model_name)\n    model, saved_config, state_report = _load_model(\n        checkpoint, repo_root, processor, model_name, device\n    )\n    rows = choose_rows(data_root, "validation", sample_count, seed)\n    if not rows:\n        raise RuntimeError("No validation records were selected")\n\n    label_counts = Counter()\n    label_total = 0\n    label_examples = []\n    label_decode_mismatches = []\n    frame_counts = Counter()\n    predictions = []\n    invalid_lengths = []\n    blank_probabilities = []\n    archive_cache = {}\n    for start in range(0, len(rows), max(1, batch_size)):\n        batch_rows = rows[start : start + max(1, batch_size)]\n        arrays = [torch.from_numpy(read_audio(row["audio"], archive_cache)) for row in batch_rows]\n        processed = processor.feature_extractor(\n            [array.numpy() for array in arrays],\n            sampling_rate=TARGET_SAMPLE_RATE,\n            return_tensors="pt",\n            padding=True,\n            return_attention_mask=True,\n        )\n        input_values = processed["input_values"].to(device)\n        attention_mask = processed["attention_mask"].to(device)\n        input_lengths = attention_mask.sum(-1).long()\n        target_lists = [processor.tokenizer(row["transcript"]).input_ids for row in batch_rows]\n        for row, target in zip(batch_rows, target_lists):\n            label_counts.update(int(item) for item in target)\n            label_total += len(target)\n            decoded_target = processor.tokenizer.decode(\n                target, group_tokens=False, skip_special_tokens=True\n            ).strip()\n            if decoded_target != row["transcript"] and len(label_decode_mismatches) < 20:\n                label_decode_mismatches.append(\n                    {\n                        "sample_id": row["sample_id"],\n                        "reference": row["transcript"],\n                        "decoded_target": decoded_target,\n                    }\n                )\n            if len(label_examples) < 5:\n                label_examples.append(\n                    {\n                        "sample_id": row["sample_id"],\n                        "reference": row["transcript"],\n                        "target_ids": target,\n                        "decoded_target": decoded_target,\n                    }\n                )\n        with torch.inference_mode():\n            outputs = model(input_values, attention_mask, input_lengths)\n        logits = outputs["logits"].float()\n        assert logits.ndim == 3\n        assert logits.shape[-1] == EXPECTED_VOCAB_SIZE\n        assert int(processor.tokenizer.pad_token_id) == EXPECTED_BLANK_ID\n        output_lengths = outputs["output_lengths"].long().clamp(0, logits.shape[1])\n        ids = logits.argmax(-1)\n        probabilities = logits.softmax(-1)\n        for row_index, row in enumerate(batch_rows):\n            length = int(output_lengths[row_index].item())\n            sequence = ids[row_index, :length].detach().cpu().tolist()\n            frame_counts.update(sequence)\n            target = target_lists[row_index]\n            repeats = sum(left == right for left, right in zip(target, target[1:]))\n            minimum = len(target) + repeats\n            if length < minimum:\n                invalid_lengths.append(\n                    {\n                        "sample_id": row["sample_id"],\n                        "output_length": length,\n                        "target_length": len(target),\n                        "adjacent_repeats": repeats,\n                        "minimum_required": minimum,\n                    }\n                )\n            probability = probabilities[row_index, :length]\n            blank_probabilities.append(float(probability[:, EXPECTED_BLANK_ID].mean().item()))\n            prediction = _decode(processor, sequence, EXPECTED_BLANK_ID)\n            predictions.append(\n                {\n                    "sample_id": row["sample_id"],\n                    "district": row["district"],\n                    "dialect": row["dialect"],\n                    "reference": row["transcript"],\n                    "prediction": prediction,\n                    "empty_prediction": not bool(prediction),\n                    "output_length": length,\n                    "target_length": len(target),\n                    "top_frame_token_ids": Counter(sequence).most_common(15),\n                }\n            )\n\n    if label_counts[EXPECTED_BLANK_ID]:\n        raise ValueError(\n            f"Valid targets contain CTC blank ID 0: count={label_counts[EXPECTED_BLANK_ID]}"\n        )\n    label_top = [[int(index), int(count)] for index, count in label_counts.most_common(20)]\n    frame_total = max(1, sum(frame_counts.values()))\n    reference_chars = sum(len(row["reference"].replace(" ", "")) for row in predictions)\n    prediction_chars = sum(len(row["prediction"].replace(" ", "")) for row in predictions)\n    cer_distance = sum(\n        edit_distance(list(row["reference"].replace(" ", "")), list(row["prediction"].replace(" ", "")))\n        for row in predictions\n    )\n    wer_distance = sum(\n        edit_distance(row["reference"].split(), row["prediction"].split())\n        for row in predictions\n    )\n    reference_words = sum(len(row["reference"].split()) for row in predictions)\n    report = {\n        "status": "ok",\n        "checkpoint": str(checkpoint),\n        "device": str(device),\n        "model_path": state_report,\n        "saved_model_num_tokens": int(saved_config.get("model", {}).get("num_tokens", -1)),\n        "ctc_contract": {\n            "blank_id": EXPECTED_BLANK_ID,\n            "padding_id": int(processor.tokenizer.pad_token_id),\n            "unknown_id": int(processor.tokenizer.unk_token_id),\n            "delimiter_id": EXPECTED_DELIMITER_ID,\n            "vocabulary_size": EXPECTED_VOCAB_SIZE,\n            "ctc_head_out_features": int(model.ctc_head.out_features),\n            "feature_sampling_rate": int(processor.feature_extractor.sampling_rate),\n            "feature_do_normalize": bool(processor.feature_extractor.do_normalize),\n            "tensor_path": "MMS encoder -> optional MoE -> ctc_head(73) -> logits -> log_softmax -> CTCLoss(blank=0)",\n        },\n        "label_audit": {\n            "valid_target_tokens": label_total,\n            "blank_id_0_count": int(label_counts[EXPECTED_BLANK_ID]),\n            "delimiter_id_2_count": int(label_counts[EXPECTED_DELIMITER_ID]),\n            "unknown_id_1_count": int(label_counts[1]),\n            "blank_fraction": label_counts[EXPECTED_BLANK_ID] / max(1, label_total),\n            "delimiter_fraction": label_counts[EXPECTED_DELIMITER_ID] / max(1, label_total),\n            "unknown_fraction": label_counts[1] / max(1, label_total),\n            "top_20_target_ids": label_top,\n            "decoded_examples": label_examples,\n            "decode_mismatches": label_decode_mismatches,\n        },\n        "length_audit": {\n            "sample_count": len(rows),\n            "invalid_count": len(invalid_lengths),\n            "invalid_samples": invalid_lengths[:50],\n            "zero_infinity_would_hide_invalid_samples": bool(invalid_lengths),\n        },\n        "raw_prediction_audit": {\n            "frame_count": int(frame_total),\n            "blank_argmax_fraction": frame_counts[EXPECTED_BLANK_ID] / frame_total,\n            "delimiter_argmax_fraction": frame_counts[EXPECTED_DELIMITER_ID] / frame_total,\n            "blank_mean_probability": sum(blank_probabilities) / max(1, len(blank_probabilities)),\n            "empty_prediction_rate": sum(item["empty_prediction"] for item in predictions) / max(1, len(predictions)),\n            "top_15_frame_token_ids": [[int(index), int(count)] for index, count in frame_counts.most_common(15)],\n            "cer": cer_distance / max(1, reference_chars),\n            "wer": wer_distance / max(1, reference_words),\n            "predictions": predictions,\n        },\n        "ctc_head_bias": _bias_report(model),\n    }\n    output_dir.mkdir(parents=True, exist_ok=True)\n    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", checkpoint.name)\n    (output_dir / f"ctc_collapse_{stem}.json").write_text(\n        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    with (output_dir / f"ctc_predictions_{stem}.csv").open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(predictions[0]))\n        writer.writeheader()\n        writer.writerows(predictions)\n    return report\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-root", type=Path, required=True)\n    parser.add_argument("--repo-root", type=Path, required=True)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--checkpoint", type=Path, action="append", default=[])\n    parser.add_argument("--sample-count", type=int, default=100)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--model-name", default="facebook/mms-300m")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--make-manifest", type=Path)\n    parser.add_argument("--manifest-count", type=int, default=32)\n    args = parser.parse_args()\n    if args.make_manifest:\n        print(json.dumps(make_manifest(args.data_root, args.make_manifest, args.manifest_count, args.seed), indent=2))\n    if not args.checkpoint:\n        if args.make_manifest:\n            return\n        parser.error("--checkpoint is required unless --make-manifest is used")\n    reports = [\n        audit_checkpoint(\n            checkpoint.resolve(),\n            args.data_root.resolve(),\n            args.repo_root.resolve(),\n            args.output_dir.resolve(),\n            args.sample_count,\n            args.batch_size,\n            args.model_name,\n            args.seed,\n        )\n        for checkpoint in args.checkpoint\n    ]\n    summary = {"checkpoints": reports}\n    (args.output_dir / "ctc_collapse_summary.json").write_text(\n        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(summary, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/kaggle_ctc_collapse_diagnostics.py': '"""Use a dataset manifest when present, otherwise use the normal file walker."""\n\nfrom __future__ import annotations\n\nimport csv\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nimport ctc_collapse_diagnostics as base\n\n\nBASE_ITER_PAIRS = base.iter_pairs\n\n\ndef _manifest_rows(root: Path, split: str):\n    manifest = root / "manifest.csv"\n    if not manifest.is_file():\n        return None\n\n    def rows():\n        with manifest.open("r", newline="", encoding="utf-8-sig") as handle:\n            for row in csv.DictReader(handle):\n                if row.get("split", "").strip().lower() != split.lower():\n                    continue\n                district = row.get("district", "").strip()\n                if district not in base.DISTRICT_TO_DIALECT:\n                    continue\n                wav_rel = row.get("relative_wav", "").replace("\\\\", "/")\n                txt_rel = row.get("relative_txt", "").replace("\\\\", "/")\n                if not wav_rel or not txt_rel:\n                    continue\n\n                wav_path = root / Path(*PurePosixPath(wav_rel).parts)\n                txt_path = root / Path(*PurePosixPath(txt_rel).parts)\n                if wav_path.is_file() and txt_path.is_file():\n                    transcript = base.normalize_text(\n                        txt_path.read_text(encoding="utf-8-sig", errors="strict")\n                    )\n                    if transcript:\n                        yield {\n                            "sample_id": row.get("sample_id") or f"{district}/{wav_path.stem}",\n                            "audio": wav_path,\n                            "transcript": transcript,\n                            "district": district,\n                            "dialect": base.DISTRICT_TO_DIALECT[district],\n                        }\n                    continue\n\n                archive_path = root / f"{split}.zip"\n                if not archive_path.is_file():\n                    continue\n                with zipfile.ZipFile(archive_path) as archive:\n                    names = {item.filename for item in archive.infolist()}\n                    if wav_rel in names and txt_rel in names:\n                        wav_name, txt_name = wav_rel, txt_rel\n                    else:\n                        wav_name = next(\n                            (name for name in names if name.endswith("/" + wav_rel)),\n                            None,\n                        )\n                        txt_name = next(\n                            (name for name in names if name.endswith("/" + txt_rel)),\n                            None,\n                        )\n                    if not wav_name or not txt_name:\n                        continue\n                    transcript = base.normalize_text(\n                        archive.read(txt_name).decode("utf-8-sig", errors="strict")\n                    )\n                    if transcript:\n                        yield {\n                            "sample_id": row.get("sample_id") or f"{district}/{wav_rel}",\n                            "audio": (archive_path, wav_name),\n                            "transcript": transcript,\n                            "district": district,\n                            "dialect": base.DISTRICT_TO_DIALECT[district],\n                        }\n\n    return rows()\n\n\ndef iter_pairs(root: Path, split: str):\n    indexed = _manifest_rows(root, split)\n    if indexed is not None:\n        yield from indexed\n    else:\n        yield from BASE_ITER_PAIRS(root, split)\n\n\nbase.iter_pairs = iter_pairs\n\n\nif __name__ == "__main__":\n    base.main()\n', 'scripts/tiny_overfit_ctc.py': '#!/usr/bin/env python\n"""Plain MMS-CTC 32-sample overfit test for the Bengali tokenizer.\n\nThe script deliberately excludes MoE, routing, dialect loss, augmentation, and\ndistributed training.  It requires a manually verified CSV manifest so a\ntranscript/audio mismatch cannot be mistaken for a model failure.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport random\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom ctc_collapse_diagnostics import (\n    EXPECTED_BLANK_ID,\n    EXPECTED_DELIMITER_ID,\n    EXPECTED_VOCAB_SIZE,\n    TARGET_SAMPLE_RATE,\n    _decode,\n    _load_processor,\n    edit_distance,\n    read_audio,\n)\n\n\ndef grad_norm(parameters) -> float:\n    import torch\n\n    total = 0.0\n    for parameter in parameters:\n        if parameter.grad is None:\n            continue\n        value = parameter.grad.detach().float()\n        if not torch.isfinite(value).all():\n            return float("nan")\n        total += float(value.pow(2).sum().item())\n    return total ** 0.5\n\n\ndef feature_output_lengths(model, input_lengths):\n    encoder = getattr(model, "wav2vec2", model)\n    return encoder._get_feat_extract_output_lengths(input_lengths).long()\n\n\ndef read_manifest(path: Path, manually_verified: bool) -> list[dict]:\n    with path.open(newline="", encoding="utf-8") as handle:\n        rows = list(csv.DictReader(handle))\n    if not rows:\n        raise ValueError("The tiny manifest is empty")\n    if len(rows) < 20 or len(rows) > 50:\n        raise ValueError(f"Tiny overfit manifest must contain 20-50 rows, got {len(rows)}")\n    if not manually_verified:\n        raise RuntimeError("Pass --manually-verified only after listening to every manifest pair")\n    unverified = [row["sample_id"] for row in rows if row.get("manually_verified", "NO").upper() != "YES"]\n    if unverified:\n        raise RuntimeError(f"Manifest rows are not manually verified: {unverified[:5]}")\n    return rows\n\n\ndef resolve_audio(value: str):\n    if "::" in value:\n        archive, member = value.split("::", 1)\n        return (Path(archive), member)\n    return Path(value)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--manifest", type=Path, required=True)\n    parser.add_argument("--checkpoint", type=Path, required=True, help="Checkpoint containing the 73-token processor")\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--model-name", default="facebook/mms-300m")\n    parser.add_argument("--max-steps", type=int, default=3000)\n    parser.add_argument("--eval-every", type=int, default=50)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--head-lr", type=float, default=2e-4)\n    parser.add_argument("--encoder-lr", type=float, default=1e-5)\n    parser.add_argument("--warmup-steps", type=int, default=50)\n    parser.add_argument("--manually-verified", action="store_true")\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    import torch\n    import torch.nn.functional as F\n    from torch.nn.utils.rnn import pad_sequence\n    from transformers import Wav2Vec2ForCTC, get_linear_schedule_with_warmup\n\n    rows = read_manifest(args.manifest.resolve(), args.manually_verified)\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    if not torch.cuda.is_available():\n        raise RuntimeError("The tiny MMS-300M overfit test requires one CUDA GPU")\n    if torch.cuda.device_count() != 1:\n        raise RuntimeError(f"Run this diagnostic on exactly one GPU, found {torch.cuda.device_count()}")\n    device = torch.device("cuda")\n    processor = _load_processor(args.checkpoint.resolve(), args.model_name)\n    feature = processor.feature_extractor\n    model = Wav2Vec2ForCTC.from_pretrained(\n        args.model_name,\n        vocab_size=EXPECTED_VOCAB_SIZE,\n        pad_token_id=EXPECTED_BLANK_ID,\n        ignore_mismatched_sizes=True,\n    )\n    model.config.ctc_loss_reduction = "mean"\n    model.config.ctc_zero_infinity = False\n    model.freeze_feature_encoder()\n    model.gradient_checkpointing_enable()\n    model.to(device).train()\n    if model.lm_head.out_features != EXPECTED_VOCAB_SIZE:\n        raise AssertionError("Tiny-test CTC head is not 73-dimensional")\n\n    head_parameters = list(model.lm_head.parameters())\n    encoder_parameters = [\n        parameter\n        for name, parameter in model.named_parameters()\n        if parameter.requires_grad and not name.startswith("lm_head.")\n    ]\n    optimizer = torch.optim.AdamW(\n        [\n            {"params": head_parameters, "lr": args.head_lr, "weight_decay": 0.0},\n            {"params": encoder_parameters, "lr": args.encoder_lr, "weight_decay": 0.0},\n        ]\n    )\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer, max(1, args.warmup_steps), max(1, args.max_steps)\n    )\n    archive_cache = {}\n    target_lists = [processor.tokenizer(row["transcript"]).input_ids for row in rows]\n    for target in target_lists:\n        if not target or EXPECTED_BLANK_ID in target:\n            raise ValueError("Tiny manifest contains an empty target or CTC blank ID")\n\n    def make_batch(batch_indices):\n        arrays = [torch.from_numpy(read_audio(resolve_audio(rows[index]["audio"]), archive_cache)) for index in batch_indices]\n        processed = feature(\n            [array.numpy() for array in arrays],\n            sampling_rate=TARGET_SAMPLE_RATE,\n            return_tensors="pt",\n            padding=True,\n            return_attention_mask=True,\n        )\n        labels = [torch.tensor(target_lists[index], dtype=torch.long) for index in batch_indices]\n        padded = pad_sequence(labels, batch_first=True, padding_value=-100)\n        lengths = torch.tensor([len(target) for target in labels], dtype=torch.long)\n        return {\n            "input_values": processed["input_values"].to(device),\n            "attention_mask": processed["attention_mask"].to(device),\n            "input_lengths": processed["attention_mask"].sum(-1).long().to(device),\n            "targets": padded.to(device),\n            "target_lengths": lengths.to(device),\n        }\n\n    def ctc_step(batch):\n        outputs = model(\n            input_values=batch["input_values"],\n            attention_mask=batch["attention_mask"],\n        )\n        logits = outputs.logits.float()\n        assert logits.ndim == 3 and logits.shape[-1] == EXPECTED_VOCAB_SIZE\n        output_lengths = feature_output_lengths(model, batch["input_lengths"])\n        flat_targets = torch.cat(\n            [labels[: int(length.item())] for labels, length in zip(batch["targets"], batch["target_lengths"])]\n        )\n        required = []\n        offset = 0\n        for length in batch["target_lengths"].tolist():\n            target = flat_targets[offset : offset + int(length)]\n            required.append(int(length) + int((target[1:] == target[:-1]).sum().item()))\n            offset += int(length)\n        required = torch.tensor(required, device=device)\n        if (output_lengths < required).any():\n            raise RuntimeError(\n                f"Invalid CTC lengths: output={output_lengths.tolist()} required={required.tolist()}"\n            )\n        loss = F.ctc_loss(\n            logits.log_softmax(-1).transpose(0, 1),\n            flat_targets,\n            output_lengths,\n            batch["target_lengths"],\n            blank=EXPECTED_BLANK_ID,\n            zero_infinity=False,\n        )\n        return loss, logits, output_lengths\n\n    def evaluate():\n        model.eval()\n        frame_ids = []\n        blank_probs = []\n        predictions = []\n        references = []\n        with torch.inference_mode():\n            for start in range(0, len(rows), max(1, args.batch_size)):\n                indices = list(range(start, min(len(rows), start + args.batch_size)))\n                batch = make_batch(indices)\n                out = model(input_values=batch["input_values"], attention_mask=batch["attention_mask"])\n                logits = out.logits.float()\n                lengths = feature_output_lengths(model, batch["input_lengths"])\n                ids = logits.argmax(-1)\n                probs = logits.softmax(-1)\n                for local, index in enumerate(indices):\n                    length = int(lengths[local].item())\n                    sequence = ids[local, :length].cpu().tolist()\n                    frame_ids.extend(sequence)\n                    blank_probs.append(float(probs[local, :length, EXPECTED_BLANK_ID].mean().item()))\n                    predictions.append(_decode(processor, sequence, EXPECTED_BLANK_ID))\n                    references.append(rows[index]["transcript"])\n        model.train()\n        blank_fraction = sum(token == EXPECTED_BLANK_ID for token in frame_ids) / max(1, len(frame_ids))\n        delimiter_fraction = sum(token == EXPECTED_DELIMITER_ID for token in frame_ids) / max(1, len(frame_ids))\n        cer_distance = sum(edit_distance(list(ref.replace(" ", "")), list(pred.replace(" ", ""))) for ref, pred in zip(references, predictions))\n        wer_distance = sum(edit_distance(ref.split(), pred.split()) for ref, pred in zip(references, predictions))\n        return {\n            "blank_fraction": blank_fraction,\n            "delimiter_fraction": delimiter_fraction,\n            "blank_mean_probability": sum(blank_probs) / max(1, len(blank_probs)),\n            "empty_prediction_rate": sum(not pred for pred in predictions) / max(1, len(predictions)),\n            "cer": cer_distance / max(1, sum(len(ref.replace(" ", "")) for ref in references)),\n            "wer": wer_distance / max(1, sum(len(ref.split()) for ref in references)),\n            "predictions": [{"reference": ref, "prediction": pred} for ref, pred in zip(references, predictions)],\n        }\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    history_path = args.output_dir / "tiny_overfit_history.jsonl"\n    best = None\n    for step in range(1, args.max_steps + 1):\n        indices = [((step - 1) * max(1, args.batch_size) + offset) % len(rows) for offset in range(max(1, args.batch_size))]\n        batch = make_batch(indices)\n        optimizer.zero_grad(set_to_none=True)\n        loss, logits, _ = ctc_step(batch)\n        if not torch.isfinite(loss).item() or not torch.isfinite(logits).all().item():\n            raise FloatingPointError("Tiny overfit produced non-finite loss or logits")\n        loss.backward()\n        head_norm = grad_norm(head_parameters)\n        encoder_norm = grad_norm(encoder_parameters)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        optimizer.step()\n        scheduler.step()\n        if step % max(1, args.eval_every) == 0 or step == 1:\n            metrics = evaluate()\n            record = {\n                "step": step,\n                "loss": float(loss.item()),\n                "head_grad_norm": head_norm,\n                "encoder_grad_norm": encoder_norm,\n                "head_lr": optimizer.param_groups[0]["lr"],\n                "encoder_lr": optimizer.param_groups[1]["lr"],\n                "loss_finite": True,\n                "logits_finite": True,\n                **metrics,\n            }\n            with history_path.open("a", encoding="utf-8") as handle:\n                handle.write(json.dumps(record, ensure_ascii=False) + "\\n")\n            print(json.dumps(record, ensure_ascii=False), flush=True)\n            if best is None or record["cer"] < best["cer"]:\n                best = record\n                torch.save(model.state_dict(), args.output_dir / "tiny_overfit_best.pt")\n\n    status = {\n        "status": "ok",\n        "best": best,\n        "passed": bool(best and best["cer"] <= 0.05 and best["wer"] <= 0.05 and best["empty_prediction_rate"] < 0.10),\n        "configuration": {\n            "samples": len(rows),\n            "batch_size": args.batch_size,\n            "max_steps": args.max_steps,\n            "eval_every": args.eval_every,\n            "model": args.model_name,\n            "moe": False,\n            "dialect_loss": False,\n            "augmentation": False,\n            "ctc_blank_id": EXPECTED_BLANK_ID,\n            "vocabulary_size": EXPECTED_VOCAB_SIZE,\n        },\n    }\n    (args.output_dir / "tiny_overfit_status.json").write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(json.dumps(status, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'src/asr_dialect_benchmark/__init__.py': '"""Research package for Bengali dialect-aware ASR."""\n', 'src/asr_dialect_benchmark/modeling/__init__.py': '"""Modeling components for the Bengali dialect ASR project."""\n\nfrom .asr_model import BengaliDialectASR\n\n__all__ = ["BengaliDialectASR"]\n', 'src/asr_dialect_benchmark/modeling/asr_model.py': '"""MMS-300M CTC baseline and dialect-aware MoE model."""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nfrom transformers import AutoModel\n\nfrom .moe import SparseMixtureOfExperts, masked_mean\n\n\ndef _value(config, name, default=None):\n    if isinstance(config, dict):\n        return config.get(name, default)\n    return getattr(config, name, default)\n\n\nclass BengaliDialectASR(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        model_config = _value(config, "model", config)\n        self.pretrained_model = _value(model_config, "pretrained_model", "facebook/mms-300m")\n        self.num_dialects = int(_value(model_config, "num_dialects", 4))\n        self.use_moe = bool(_value(model_config, "use_moe", True))\n        self.encoder = AutoModel.from_pretrained(self.pretrained_model)\n        if bool(_value(model_config, "gradient_checkpointing", True)):\n            self.encoder.gradient_checkpointing_enable()\n        hidden_size = int(self.encoder.config.hidden_size)\n        if self.use_moe:\n            self.moe = SparseMixtureOfExperts(\n                hidden_size=hidden_size,\n                num_dialects=self.num_dialects,\n                top_k=int(_value(model_config, "top_k", 2)),\n                dropout=float(_value(model_config, "dropout", 0.1)),\n                use_router=bool(_value(model_config, "use_router", True)),\n                use_shared_expert=bool(_value(model_config, "use_shared_expert", True)),\n            )\n        else:\n            self.moe = None\n        self.dialect_classifier = nn.Linear(hidden_size, self.num_dialects)\n        self.ctc_head = nn.Linear(hidden_size, int(_value(model_config, "num_tokens", 64)))\n\n    def feature_lengths(self, input_lengths: torch.Tensor) -> torch.Tensor:\n        return self.encoder._get_feat_extract_output_lengths(input_lengths).to(torch.long)\n\n    def set_phase(self, phase: int, top_layers: int = 4) -> None:\n        for parameter in self.encoder.parameters():\n            parameter.requires_grad = False\n        if phase >= 2:\n            layers = self.encoder.encoder.layers\n            for layer in layers[-top_layers:]:\n                for parameter in layer.parameters():\n                    parameter.requires_grad = True\n            # The final normalization is part of the top representation.\n            if hasattr(self.encoder.encoder, "layer_norm"):\n                for parameter in self.encoder.encoder.layer_norm.parameters():\n                    parameter.requires_grad = True\n\n    def forward(self, input_values=None, attention_mask=None, input_lengths=None, routing_inputs=None):\n        if routing_inputs is not None:\n            if self.moe is None:\n                return {"gate_probs": None, "topk_indices": None}\n            gate_probs, _, topk_indices = self.moe.route(routing_inputs)\n            return {"gate_probs": gate_probs, "topk_indices": topk_indices}\n        encoded = self.encoder(input_values=input_values, attention_mask=attention_mask)\n        hidden_states = encoded.last_hidden_state\n        if input_lengths is None:\n            input_lengths = attention_mask.sum(-1) if attention_mask is not None else input_values.new_full((input_values.shape[0],), input_values.shape[1], dtype=torch.long)\n        output_lengths = self.feature_lengths(input_lengths)\n        output_lengths = output_lengths.clamp(max=hidden_states.shape[1])\n        time = torch.arange(hidden_states.shape[1], device=hidden_states.device)[None, :]\n        feature_mask = time < output_lengths[:, None]\n        if self.moe is not None:\n            hidden_states, gate_probs, topk_indices, router_input = self.moe(hidden_states, feature_mask)\n        else:\n            gate_probs, topk_indices, router_input = None, None, None\n        pooled = masked_mean(hidden_states, feature_mask)\n        return {\n            "logits": self.ctc_head(hidden_states),\n            "dialect_logits": self.dialect_classifier(pooled),\n            "gate_probs": gate_probs,\n            "topk_indices": topk_indices,\n            "router_input": router_input,\n            "output_lengths": output_lengths,\n        }\n', 'src/asr_dialect_benchmark/modeling/experts.py': 'import torch\nimport torch.nn as nn\n\n\nclass ResidualFeedForward(nn.Module):\n    """Lightweight residual feed-forward expert block."""\n\n    def __init__(self, hidden_size: int, ff_dim: int = 512, dropout: float = 0.1):\n        super().__init__()\n        self.fc1 = nn.Linear(hidden_size, ff_dim)\n        self.act = nn.GELU()\n        self.dropout = nn.Dropout(dropout)\n        self.fc2 = nn.Linear(ff_dim, hidden_size)\n        self.norm = nn.LayerNorm(hidden_size)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        residual = x\n        x = self.fc1(x)\n        x = self.act(x)\n        x = self.dropout(x)\n        x = self.fc2(x)\n        x = self.dropout(x)\n        return self.norm(residual + x)\n\n\nclass SharedExpert(nn.Module):\n    def __init__(self, hidden_size: int, ff_dim: int = 512, dropout: float = 0.1):\n        super().__init__()\n        self.block = ResidualFeedForward(hidden_size, ff_dim=ff_dim, dropout=dropout)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.block(x)\n\n\nclass DialectExpert(nn.Module):\n    def __init__(self, hidden_size: int, ff_dim: int = 512, dropout: float = 0.1):\n        super().__init__()\n        self.block = ResidualFeedForward(hidden_size, ff_dim=ff_dim, dropout=dropout)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.block(x)\n', 'src/asr_dialect_benchmark/modeling/moe.py': '"""Utterance-routed sparse dialect mixture of experts."""\n\nimport torch\nimport torch.nn as nn\n\nfrom .experts import DialectExpert, SharedExpert\nfrom .router import DialectRouter\n\n\ndef masked_mean(hidden_states: torch.Tensor, mask: torch.Tensor | None) -> torch.Tensor:\n    if mask is None:\n        return hidden_states.mean(dim=1)\n    weights = mask.to(hidden_states.dtype).unsqueeze(-1)\n    return (hidden_states * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)\n\n\nclass SparseMixtureOfExperts(nn.Module):\n    def __init__(self, hidden_size: int, num_dialects: int = 4, top_k: int = 2, dropout: float = 0.1, use_router: bool = True, use_shared_expert: bool = True):\n        super().__init__()\n        self.use_router = use_router\n        self.use_shared_expert = use_shared_expert\n        self.top_k = min(top_k, num_dialects)\n        self.router = DialectRouter(hidden_size, num_dialects=num_dialects, dropout=dropout) if use_router else None\n        self.shared_expert = SharedExpert(hidden_size=hidden_size, dropout=dropout) if use_shared_expert else None\n        self.dialect_experts = nn.ModuleList(DialectExpert(hidden_size=hidden_size, dropout=dropout) for _ in range(num_dialects))\n\n    def route(self, pooled: torch.Tensor):\n        if self.router is None:\n            gate_probs = pooled.new_full((pooled.shape[0], len(self.dialect_experts)), 1.0 / len(self.dialect_experts))\n        else:\n            gate_probs = self.router(pooled)\n        topk_values, topk_indices = torch.topk(gate_probs, self.top_k, dim=-1)\n        topk_values = topk_values / topk_values.sum(dim=-1, keepdim=True).clamp_min(1e-9)\n        return gate_probs, topk_values, topk_indices\n\n    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor | None = None):\n        pooled = masked_mean(hidden_states, attention_mask)\n        gate_probs, topk_values, topk_indices = self.route(pooled)\n        fusion = torch.zeros_like(pooled)\n        # Dispatch only selected samples to each expert. Top-1 therefore\n        # performs half the dialect-expert work of top-2.\n        for expert_id, expert in enumerate(self.dialect_experts):\n            sample_indices, slots = torch.where(topk_indices == expert_id)\n            if sample_indices.numel() == 0:\n                continue\n            expert_output = expert(pooled.index_select(0, sample_indices))\n            weighted = expert_output * topk_values[sample_indices, slots].unsqueeze(-1)\n            fusion = fusion.index_add(0, sample_indices, weighted)\n        if self.shared_expert is not None:\n            fusion = fusion + self.shared_expert(pooled)\n        return hidden_states + fusion.unsqueeze(1), gate_probs, topk_indices, pooled\n', 'src/asr_dialect_benchmark/modeling/router.py': 'import torch\nimport torch.nn as nn\n\n\nclass DialectRouter(nn.Module):\n    """Predicts dialect probabilities over four dialects."""\n\n    def __init__(self, input_dim: int, num_dialects: int = 4, dropout: float = 0.1):\n        super().__init__()\n        self.input_dim = input_dim\n        self.num_dialects = num_dialects\n        self.proj1 = nn.Linear(input_dim, max(64, input_dim // 2))\n        self.act = nn.GELU()\n        self.dropout = nn.Dropout(dropout)\n        self.proj2 = nn.Linear(max(64, input_dim // 2), num_dialects)\n\n    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:\n        if hidden_states.dim() == 3:\n            x = hidden_states.mean(dim=1)\n        else:\n            x = hidden_states\n        x = self.proj1(x)\n        x = self.act(x)\n        x = self.dropout(x)\n        x = self.proj2(x)\n        return torch.softmax(x, dim=-1)\n'}

for relative_name, source_text in EMBEDDED_SOURCES.items():
    destination = REPO_DIR / relative_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source_text, encoding="utf-8")
print("Embedded diagnostic source snapshot:", REPO_COMMIT)
print("No HF token is required for this notebook; the MMS model is public.")

INPUT_ROOT = Path("/kaggle/input")
DATASET_SLUG = "four-dialect-data-undersampled"
def is_dataset_root(path: Path) -> bool:
    return all((path / split).is_dir() or (path / f"{split}.zip").is_file() for split in ("train", "validation", "test"))

# Prefer the exact Kaggle mount before using a shallow fallback.  This avoids
# recursively walking every WAV file in the attached dataset.
known_data_roots = [
    INPUT_ROOT / "datasets" / "diyalibiswas" / "four-dialect-data-undersampled",
    INPUT_ROOT / "four-dialect-data-undersampled",
    INPUT_ROOT / "four_dialect_data_undersampled",
]
dataset_candidates = [path for path in known_data_roots if path.is_dir() and is_dataset_root(path)]
if not dataset_candidates:
    dataset_candidates = [
        path for path in INPUT_ROOT.iterdir()
        if path.is_dir() and DATASET_SLUG in str(path).lower() and is_dataset_root(path)
    ]
if not dataset_candidates:
    dataset_candidates = [
        path for path in INPUT_ROOT.iterdir()
        if path.is_dir() and is_dataset_root(path)
    ]
if not dataset_candidates:
    raise FileNotFoundError("Attach diyalibiswas/four-dialect-data-undersampled to notebook input")
DATA_ROOT = sorted(dataset_candidates, key=lambda path: len(str(path)))[0]
print("DATA_ROOT=", DATA_ROOT)

CHECKPOINT_OVERRIDE = os.environ.get("CTC_CHECKPOINT", "").strip()
CHECKPOINT_ROOT = Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else None
if CHECKPOINT_ROOT and not CHECKPOINT_ROOT.exists():
    raise FileNotFoundError(CHECKPOINT_ROOT)
if CHECKPOINT_ROOT is None:
    checkpoint_candidates = []
    known_output_roots = [
        INPUT_ROOT / "datasets" / "diyalibiswas" / "output",
        INPUT_ROOT / "output",
    ]
    for output_root in [path for path in known_output_roots if path.is_dir()]:
        for config_path in output_root.rglob("config.json"):
            candidate = config_path.parent
            if any((candidate / state_name).is_file() for state_name in ("model.safetensors", "model_state.pt", "pytorch_model.bin")):
                checkpoint_candidates.append(candidate)
    if not checkpoint_candidates:
        raise FileNotFoundError(
            "No checkpoint found below /kaggle/input/datasets/diyalibiswas/output; attach the output dataset"
        )
    def checkpoint_step(path):
        state_path = path / "trainer_state.json"
        if not state_path.is_file():
            return -1
        try:
            return int(json.loads(state_path.read_text(encoding="utf-8")).get("global_step", -1))
        except Exception:
            return -1
    CHECKPOINT_ROOT = sorted(
        checkpoint_candidates,
        key=lambda path: (checkpoint_step(path), str(path)),
    )[-1]
print("CHECKPOINT_ROOT=", CHECKPOINT_ROOT)


In [ ]:
# Build the deterministic 32-row manifest and record the user's explicit
# confirmation that every selected audio/transcript pair was checked.
manifest_path = RUN_DIR / "tiny_manifest.csv"
manifest_command = [
    sys.executable, str(REPO_DIR / "scripts" / "kaggle_ctc_collapse_diagnostics.py"),
    "--data-root", str(DATA_ROOT), "--repo-root", str(REPO_DIR),
    "--output-dir", str(RUN_DIR), "--make-manifest", str(manifest_path),
    "--manifest-count", "32",
]
subprocess.run(manifest_command, check=True)

import csv
with manifest_path.open("r", newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    fieldnames = list(reader.fieldnames or [])
    rows = list(reader)
if len(rows) != 32:
    raise RuntimeError(f"Expected exactly 32 manifest rows, got {len(rows)}")
for row in rows:
    row["manually_verified"] = "YES"
with manifest_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
print(f"Recorded confirmed audio/transcript matches for {len(rows)} rows")
print(manifest_path.read_text(encoding="utf-8")[:1500])


In [ ]:
# Audit the latest attached checkpoint. No training is launched.
audit_log = LOG_DIR / "checkpoint_audit.log"
if CHECKPOINT_ROOT is None:
    status = {
        "status": "blocked",
        "reason": "No attached checkpoint dataset was found",
        "action": "Attach the Kaggle output dataset containing checkpoint files and rerun this cell",
    }
    (RUN_DIR / "checkpoint_audit_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    print(json.dumps(status, indent=2))
else:
    command = [
        sys.executable, str(REPO_DIR / "scripts" / "ctc_collapse_diagnostics.py"),
        "--data-root", str(DATA_ROOT), "--repo-root", str(REPO_DIR),
        "--output-dir", str(RUN_DIR), "--checkpoint", str(CHECKPOINT_ROOT),
        "--sample-count", "100", "--batch-size", "4",
    ]
    with audit_log.open("w", encoding="utf-8") as handle:
        completed = subprocess.run(command, stdout=handle, stderr=subprocess.STDOUT, text=True)
    if completed.returncode:
        print("--- checkpoint_audit.log (tail) ---")
        print(audit_log.read_text(encoding="utf-8", errors="replace")[-20000:])
        raise RuntimeError(f"Checkpoint audit failed; inspect {audit_log}")
    print((RUN_DIR / "ctc_collapse_summary.json").read_text(encoding="utf-8"))


In [ ]:
# Run the one-GPU plain MMS-CTC overfit test. CUDA_VISIBLE_DEVICES=0 makes
# this safe even when Kaggle provisions a T4x2 session.
if CHECKPOINT_ROOT is None:
    raise RuntimeError("No checkpoint was found in the attached output dataset")
tiny_command = [
    sys.executable, str(REPO_DIR / "scripts" / "tiny_overfit_ctc.py"),
    "--manifest", str(manifest_path), "--checkpoint", str(CHECKPOINT_ROOT),
    "--output-dir", str(RUN_DIR / "tiny-overfit"), "--batch-size", "4",
    "--max-steps", "3000", "--eval-every", "50", "--manually-verified",
]
tiny_env = os.environ.copy()
tiny_env["CUDA_VISIBLE_DEVICES"] = "0"
print("Running:", " ".join(map(str, tiny_command)))
completed = subprocess.run(tiny_command, check=False, env=tiny_env)
if completed.returncode:
    raise RuntimeError(f"Tiny overfit failed with exit code {completed.returncode}")
status_path = RUN_DIR / "tiny-overfit" / "tiny_overfit_status.json"
print(status_path.read_text(encoding="utf-8"))


In [ ]:
# Save the confirmed manifest, metrics, logs, and status as one Kaggle output.
manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repository_commit": REPO_COMMIT,
    "dataset_root": str(DATA_ROOT),
    "checkpoint_root": str(CHECKPOINT_ROOT) if CHECKPOINT_ROOT else None,
    "confirmed_manifest": str(manifest_path),
    "tiny_overfit": True,
    "model_initialized_fresh": True,
    "gpu_visibility": "0",
}
(RUN_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive = Path("/kaggle/working/ctc-confirmed-tiny-overfit-outputs.zip")
if archive.exists():
    archive.unlink()
shutil.make_archive(str(archive.with_suffix("")), "zip", RUN_DIR)
print("Saved:", archive)
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN_DIR), path.stat().st_size)
